# 02 · Primeiros Passos com PySpark

🎯 **Objetivo:** Ganhar fluência nos verbos fundamentais do PySpark — `select`, `filter`, `withColumn`, `orderBy`, os métodos de saída `show`/`collect`/`toPandas`, e a persistência de dados (`write`) em Parquet, Parquet particionado, CSV e JSON.

**Teoria:** docs/05-pyspark-na-pratica.md

Sem Docker — `local[*]` roda Driver e Executors como threads neste mesmo processo. Este notebook trabalha com um dataset de negócio real: `empresas`, `funcionarios` e `vendas`.

📌 O objetivo aqui **não** é entender COMO o Spark distribui trabalho (isso vem em notebooks futuros) — é ganhar prática nos verbos que você vai usar o tempo todo.

---
### 🔤 As seis famílias de operações que você vai aprender

1. **Projeção** — `select`, `withColumn`, `drop`
2. **Filtragem** — `filter` / `where`
3. **Ordenação** — `orderBy` / `sort`
4. **Exploração** — `distinct`, `describe`
5. **Saída (Driver)** — `show`, `first`, `collect`, `toPandas`
6. **Persistência (disco)** — `write` em Parquet, Parquet particionado, CSV e JSON

Todas elas se aplicam sobre DataFrame, que é a abstração central do Spark SQL.
Vamos começar criando a SparkSession.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

spark

✅ **SparkSession criada com sucesso.** Repare na saída: o Spark exibe informações da versão (`3.5.x`), o nome da aplicação e o mestre (`local[*]`, que significa "use todos os núcleos da máquina").

## Lendo o dataset

Rode `make generate-data SCALE=small` antes, se ainda não gerou os dados.

📌 Estamos com Spark no modo local/Standalone.

In [ ]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("../data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("../data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("../data/bronze/vendas")

# count() é uma AÇÃO — força o Spark a processar os dados e traz o total de linhas
# printSchema() mostra a estrutura (colunas e tipos) sem trazer todos os dados
print(f"{sdf_empresas.count():,} linhas")
sdf_empresas.printSchema()

print(f"{sdf_funcionarios.count():,} linhas")
sdf_funcionarios.printSchema()

print(f"{sdf_vendas.count():,} linhas")
sdf_vendas.printSchema()

📌 **Observações sobre a saída:**

- `count()` é uma **ação** — ela força o Spark a percorrer todo o dataset e computar o resultado. Sem ela, o Spark só guardaria o plano.
- `printSchema()` é uma **ação de metadados** — ela lê o esquema do Parquet (que já está embutido nos arquivos) sem precisar escanear todas as linhas.
- Repare nos tipos: `long`, `double`, `string`, `date` — cada coluna tem um tipo bem definido, diferente do Python puro.

💡 **Dica:** Parquet é o formato padrão do Spark. Ele armazena dados em colunas (não linhas), o que acelera consultas que usam poucas colunas.


## Inspecionando os Spark Data Frames

Todo Spark Data Frame (SDF) possui vários métodos python. Vamos usar aqui três deles:

- `.printSchema()` - imprimir tipagem dos campos
- `.show()` - mostrar 20 primeiras linhas
- `.count()` - total de linhas do SDF

In [ ]:
sdf_vendas.show()

In [ ]:
sdf_vendas.printSchema()

In [ ]:
sdf_funcionarios.show(5)

In [ ]:
sdf_empresas.count()

## `select` e `withColumn`

`select` escolhe colunas existentes (projeção). Já `withColumn` cria uma **nova coluna** (ou substitui uma existente) a partir de uma expressão sobre as demais.

🧠 **Por quê?** No dia a dia você vai: 
- (1) selecionar apenas as colunas relevantes para reduzir o volume de dados, e 
- (2) derivar novas colunas com transformações de negócio.



In [ ]:
# inspeção inicial
sdf_funcionarios.show()

In [ ]:
sdf_funcionarios.printSchema()

#### 💡 **Exemplo 1:** Calcular o salário anual a partir do salário mensal.

Vamos importar a função `col` do pyspark sql. Ela vai ajudar a selecionar a coluna-alvo para o método de criação/atualização `.withColumn`.

Montando o plano lógico  (lazy evaluation)

In [ ]:
from pyspark.sql.functions import col

# Pipeline de transformação: select -> withColumn
# Nada é executado ainda — só montamos o plano lógico (lazy evaluation)
funcionarios_resumo = (
    sdf_funcionarios
    # Projeta apenas 3 colunas das dezenas disponíveis
    .select("nome_funcionario", "cargo", "salario")
    # Cria nova coluna calculada: salário mensal x 12 meses
    .withColumn("salario_anual", col("salario") * 12)
)

Executando a ação.

In [ ]:
funcionarios_resumo.show()

📌 **Entendendo a saída:**

- A coluna `salario_anual` foi calculada como `salario * 12`.
- O Spark suporta operações aritméticas diretamente via `col()`.
- O pipeline `select().withColumn()` é **lento** (lazy) — nada executa até o `show()`.

⚠️ **Atenção:** `withColumn` **não modifica** o DataFrame original. Ele retorna um **novo** DataFrame. 

**Assim como RDDs, os DataFrames são imutáveis no Spark**.


In [ ]:
sdf_funcionarios.show()

#### 💡 **Exemplo 2:** Calcular a coluna salário anual em logaritmo e atualizar a coluna nome para caracteres maiúsculos

In [ ]:
from pyspark.sql.functions import col, log, upper

# Pipeline lazy
func_resumo = (
    sdf_funcionarios
    .select("nome_funcionario", "cargo", "salario")
    # Cria nova coluna calculada: salário mensal x 12 meses
    .withColumn("salario_anual_log", log(col("salario") * 12))
    .withColumn("nome_funcionario", upper(col("nome_funcionario")))
)

In [ ]:
# Executar plano
func_resumo.show()

#### 💡 **Exemplo 3:** Classificar funcionários por faixa salarial com `udf` e `pandas_udf`

Nem toda regra de negócio existe pronta em `pyspark.sql.functions`. Quando a lógica é condicional ou vem de uma biblioteca Python qualquer, usamos uma **UDF (User Defined Function)** — uma função Python que o Spark passa a executar sobre o DataFrame.

Existem dois estilos:

1. **UDF tradicional** (`udf`) — roda **linha a linha**. Cada valor é serializado (pickle), enviado a um processo Python, processado e o resultado volta serializado. Simples de escrever, porém lento em datasets grandes.
2. **`pandas_udf`** (UDF vetorizada) — usa **Apache Arrow** para trocar dados em **lotes (batches)** como `pandas.Series`, aproveitando operações vetorizadas do NumPy/pandas. Muito mais rápida — é a forma recomendada quando a lógica não existe em `pyspark.sql.functions`.

Vamos classificar `sdf_funcionarios` em faixas salariais (`Junior`, `Pleno`, `Senior`, `Executivo`) com os dois métodos.


In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


# Função Python "pura" — nada aqui sabe que vai rodar dentro do Spark
def classificar_faixa_salarial(salario: float) -> str:
    if salario < 4000:
        return "Junior"
    elif salario < 8000:
        return "Pleno"
    elif salario < 12000:
        return "Senior"
    else:
        return "Executivo"


# Registra a função Python como UDF do Spark, declarando o tipo de retorno
faixa_salarial_udf = udf(classificar_faixa_salarial, StringType())

# Pipeline lazy — o Spark ainda não executou a UDF sobre nenhuma linha
funcionarios_faixa = sdf_funcionarios.withColumn(
    "faixa_salarial", faixa_salarial_udf(col("salario"))
).select("nome_funcionario", "salario", "faixa_salarial")

In [ ]:
# Executando a ação — aqui o Spark chama a UDF uma vez POR LINHA,
# serializando cada valor de "salario" para o processo Python e de volta
funcionarios_faixa.show(10)

📌 **O problema da UDF tradicional:** ela é uma "caixa preta" para o otimizador Spark (Catalyst não enxerga o que acontece dentro dela) e processa **uma linha por vez**, com o custo de serializar/desserializar cada valor entre a JVM e o processo Python. Em datasets grandes, isso vira um gargalo real.

Agora vamos resolver o mesmo problema com `pandas_udf`, que troca dados em **lotes** via Apache Arrow.


In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf


# Mesma regra de negócio, mas vetorizada: recebe e devolve um pandas.Series
# (um LOTE de valores, não um valor por vez)
@pandas_udf(StringType())
def faixa_salarial_pandas_udf(salario: pd.Series) -> pd.Series:
    condicoes = [salario < 4000, salario < 8000, salario < 12000]
    faixas = ["Junior", "Pleno", "Senior"]
    # np.select aplica as condições em bloco — sem loop Python linha a linha
    aplica = np.select(condicoes, faixas, default="Executivo")
    return pd.Series(aplica, index=salario.index)


funcionarios_faixa_pandas = sdf_funcionarios.withColumn(
    "faixa_salarial", faixa_salarial_pandas_udf(col("salario"))
).select("nome_funcionario", "salario", "faixa_salarial")

In [ ]:
# Executando a ação — o Spark envia colunas inteiras (em lotes/Arrow) para o
# processo Python, que responde com um pandas.Series por lote, não linha a linha
funcionarios_faixa_pandas.show(10)

📌 **`udf` vs. `pandas_udf` — quando usar cada uma:**

| | `udf` | `pandas_udf` |
|---|---|---|
| Unidade de processamento | uma linha por vez | um lote (`pandas.Series`) por vez |
| Transporte de dados | pickle, linha a linha | Apache Arrow, colunar e em lote |
| Desempenho | mais lento, cresce mal com volume | muito mais rápido — aproveita vetorização do NumPy/pandas |
| Quando usar | protótipos rápidos, lógica muito simples | qualquer lógica de negócio que rode em produção |

⚠️ **Pré-requisito:** `pandas_udf` depende do pacote `pyarrow` instalado no ambiente (`pip install pyarrow`) para o transporte via Arrow funcionar.

🧠 **Regra prática:** se a transformação já existe em `pyspark.sql.functions`, use-a — ela roda nativamente na JVM e é mais rápida que qualquer UDF. UDFs (de preferência `pandas_udf`) ficam reservadas para lógica que **não tem equivalente nativo**.


## `filter`

Duas perguntas de negócio simples que o `filter` responde:

1. **Quais vendas foram grandes?** (> R\$500)
2. **Quais funcionários foram contratados recentemente?** (a partir de 2024-07-01)

🧠 **Por quê?** `filter` (ou `where` — são sinônimos) é a operação mais usada para reduzir dados a apenas o que interessa para a análise. O Spark aplica o filtro o mais cedo possível no plano de execução (**predicate pushdown**) para minimizar o tráfego de dados.

In [ ]:
# Filtra vendas com valor superior a R$ 500
vendas_grandes = sdf_vendas.filter(col("valor") > 500)
# count() executa o plano e retorna o total de linhas filtradas
print(f"Vendas acima de R$500: {vendas_grandes.count():,} de {sdf_vendas.count():,}")

# Filtra funcionários admitidos a partir de julho de 2024
contratados_recentes = sdf_funcionarios.filter(col("data_admissao") >= "2024-07-01")
print(f"Funcionários admitidos desde 2024-07-01: {contratados_recentes.count():,}")
# Encadeia filter com select e show para ver apenas colunas relevantes
contratados_recentes.select("nome_funcionario", "cargo", "data_admissao").show(10)

📌 **Interpretação dos resultados:**

- A primeira contagem mostra quantas vendas são consideradas "grandes" (> R\$500) em relação ao total.
- A segunda mostra quantos funcionários foram contratados recentemente.
- Perceba como `filter().select().show()` encadeia três operações em uma linha — isso é o **estilo funcional** do Spark.

💡 **Dica:** Você pode usar `where()` no lugar de `filter()` — é um alias, o comportamento é idêntico.


In [ ]:
# where() é um alias de filter() — provando que o resultado é idêntico
vendas_grandes_where = sdf_vendas.where(col("valor") > 500)

vendas_grandes_where.count() == vendas_grandes.count()

#### 💡 **Exemplo 4:** Filtrar funcionários — salário acima da média do próprio cargo, entre os "Vendedor Pleno"

⚠️ **Cuidado:** `avg()` é uma função de **agregação** — ela reduz várias linhas a um único valor. Por isso não pode ser usada diretamente dentro de `filter()`/`where()`, que avalia uma condição **linha a linha** contra um valor escalar. Primeiro calculamos a média como um número Python e só então a usamos no filtro.

📌 Note que calculamos a média **apenas entre os "Vendedor Pleno"**, não a média geral da empresa — comparar o salário de um cargo júnior/pleno contra a média de todos os cargos (que inclui gerentes) não diria muito sobre quem se destaca dentro do próprio grupo.


In [ ]:
from pyspark.sql.functions import avg, col

# avg() é uma agregação — reduz várias linhas a UM valor.
# Filtramos por cargo ANTES de agregar, para obter a média salarial só dos "Vendedor Pleno"
# select() + first() traz esse valor para o Driver como um float Python comum
salario_medio_pleno = (
    sdf_funcionarios
    .filter(col("cargo") == "Vendedor Pleno")
    .select(avg(col("salario")))
).first()[0]

print(f"Salário médio (Vendedor Pleno): R$ {salario_medio_pleno:,.2f}")

# Agora comparamos cada linha com o valor escalar (já não é mais uma agregação)
# orderBy(desc) traz quem ganha mais no topo — os "destaques" do cargo
resumo = (
    sdf_funcionarios
    .filter(col("cargo") == "Vendedor Pleno")
    .filter(col("salario") > salario_medio_pleno)
    .select("nome_funcionario", "salario")
    .orderBy(col("salario").desc())
)

resumo.show()

## `drop`, `distinct`, `describe`

Três operações úteis para explorar e limpar dados:

- `distinct()` — remove linhas duplicadas (útil para descobrir valores únicos)
- `describe()` — estatísticas descritivas (count, mean, stddev, min, max)
- `drop()` — remove uma coluna do DataFrame

💡 **Dica:** Use `distinct()` em colunas categóricas para entender a cardinalidade dos seus dados antes de fazer agregações. Combine com `orderBy()` para listar os valores únicos em ordem, em vez da ordem "aleatória" que o Spark devolve por padrão.

**Quais os setores das empresas?**: usamos `distinct` + `orderBy` (para listar em ordem alfabética).

In [ ]:
# Valores únicos de setor, ordenados alfabeticamente com orderBy
# truncate=False garante que textos longos não sejam cortados
setores_unicos = sdf_empresas.select("setor").distinct().orderBy("setor")
setores_unicos.show(truncate=False)

**Quais os cargos dos funcionários?**: usamos `distinct` + `orderBy` (por padrão, ordem ascendente).

In [ ]:

# Cargos únicos — quantos cargos diferentes existem na empresa?
cargos_unicos = sdf_funcionarios.select("cargo").distinct().orderBy("cargo")
cargos_unicos.show(truncate=False)


**Quais as estatísticas descritivas das vendas?**:  usamos `describe`.

In [ ]:
# Estatísticas descritivas da coluna valor (média, desvio, min, max)
sdf_vendas.select("valor").describe().show()


#### 💡 **Exemplo 5:** Estatísticas customizadas com `pandas_udf` (mediana e assimetria)

`describe()` só devolve `count`, `mean`, `stddev`, `min` e `max` — não há `median()` nem `skew()` (assimetria) nativos em `pyspark.sql.functions`. 

Uma **pandas UDF do tipo Series → escalar** resolve isso: ela recebe a coluna inteira (em lotes) como `pandas.Series` e devolve **um único valor**, funcionando como uma agregação — igual `sum()` ou `avg()`. Dentro dela podemos usar qualquer método do pandas/NumPy/SciPy.


In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf


# Series-to-scalar: recebe a coluna inteira e devolve UM valor — o Spark trata
# como uma agregação, não como uma transformação linha a linha
@pandas_udf("double")
def mediana(valores: pd.Series) -> float:
    return valores.median()


@pandas_udf("double")
def assimetria(valores: pd.Series) -> float:
    # skew() mede o quanto a distribuição é assimétrica — não existe em pyspark.sql.functions
    return valores.skew()


# Chamamos as duas pandas UDFs dentro de select(), exatamente como faríamos com avg()
sdf_vendas.select(
    mediana(col("valor")).alias("mediana_valor"),
    assimetria(col("valor")).alias("assimetria_valor"),
).show()

📌 **Entendendo a saída:**

- A `mediana_valor` (~R\$54,68) fica bem abaixo da `mean` (~R\$90,22) mostrada no `describe()` — sinal de que a distribuição não é simétrica.
- A `assimetria_valor` positiva e alta confirma isso: a coluna `valor` tem uma **cauda longa à direita** (muitas vendas baixas e poucas vendas bem altas puxando a média para cima).

🧠 **Por quê usar `pandas_udf` aqui e não uma UDF tradicional?** Estatísticas como mediana e assimetria exigem enxergar **todos os valores da coluna de uma vez** (não dá pra calcular linha a linha). O `pandas_udf` do tipo Series-to-scalar entrega exatamente isso — lotes vetorizados de `pandas.Series` — enquanto reaproveita toda a biblioteca pandas/NumPy/SciPy que já existe pronta.

**Exemplo**: Criamos um novo sdf de funcionarios sem a coluna id_funcionario

In [ ]:

# Remove a coluna id_funcionario do DataFrame (não altera o original)
funcionarios_sem_id = sdf_funcionarios.drop("id_funcionario")
funcionarios_sem_id.show(3)

📌 **Resumo das operações:**

- `distinct()` revelou quantos setores e cargos diferentes existem nos dados — informação valiosa antes de fazer agregações.
- `describe()` forneceu média, desvio padrão, mínimo e máximo da coluna `valor` — um resumo estatístico rápido.
- `drop()` removeu a coluna `id_funcionario` sem alterar o DataFrame original.

🧠 **Por quê** `drop` não altera o original? DataFrames Spark são **imutáveis**. Toda transformação retorna um novo DataFrame.


## `show` vs. `collect` vs. `toPandas`

Cada método de saída tem um propósito e um custo diferente:

| Método | Retorno | Quando usar |
|---|---|---|
| `show()` | `None` (só imprime) | Dar uma olhada rápida nos dados |
| `collect()` | `list[Row]` | Pequenos resultados para processar em Python puro |
| `toPandas()` | `pd.DataFrame` | Plotar, exportar ou integrar com Pandas/ML |

⚠️ **Atenção:** `collect()` e `toPandas()` trazem **todos os dados para a memória do Driver**. Com datasets grandes, isso pode causar `OutOfMemoryError`. Sempre use `.limit()` ou `.filter()` antes.

In [ ]:
# Pega apenas 5 linhas como amostra (evita trazer tudo para o Driver)
amostra = sdf_empresas.limit(5)

# show(): imprime na tela, não retorna nada utilizável
amostra.show()
# collect(): retorna uma lista Python de objetos Row
linhas = amostra.collect()
print(type(linhas), linhas[0])

# toPandas(): converte para pandas DataFrame (requer que os dados caibam na RAM)
df = amostra.toPandas()
print(type(df))
# Exibe o pandas DataFrame com formatação rica
df

📌 **Quando usar cada um?**

- **`show()`** — uso diário para dar uma espiada nos dados
- **`collect()`** — bom para resultados pequenos que você quer processar com lógica Python (ex: loops, condicionais)
- **`toPandas()`** — ideal para gerar gráficos com matplotlib/seaborn ou exportar para CSV/Excel

⚠️ **Regra de ouro:** Nunca faça `collect()` ou `toPandas()` em um DataFrame inteiro sem antes filtrar ou limitar. Sempre pergunte: *quanto disso cabe na RAM do meu laptop?*


#### 💡 **Exemplo 6:** Ordenando com `orderBy` — do `show()` ao `toPandas()`

`orderBy` (sinônimo: `sort`) ordena as linhas de um DataFrame por uma ou mais colunas. Por padrão a ordenação é **ascendente**; use `col(...).desc()` para inverter. Dá pra ordenar por várias colunas ao mesmo tempo, cada uma com sua própria direção.

⚠️ Assim como `filter`, `orderBy` é uma transformação **lazy** — só ordena de fato quando uma ação (`show`, `collect`, `toPandas`, `count`...) dispara o plano. E como ordenar o dataset inteiro é uma operação cara (exige embaralhar dados entre partições — **shuffle**), sempre combine `orderBy` com `limit()` **antes** de trazer o resultado para o Driver.

In [ ]:
# As 5 vendas de maior valor — orderBy(desc) + limit ANTES de trazer para o Driver
top_vendas = (
    sdf_vendas
    .orderBy(col("valor").desc())
    .limit(5)
)

# show(): só para dar uma olhada rápida — não retorna nada utilizável
top_vendas.show()

In [ ]:
# As mesmas 5 vendas, agora como objetos Row do Python via collect()
# Como já limitamos a 5 linhas antes do collect(), não há risco de estourar a memória do Driver
top_vendas_rows = top_vendas.collect()

for linha in top_vendas_rows:
    print(f"Venda #{linha.id_venda}: R$ {linha.valor:,.2f} (empresa {linha.id_empresa})")

In [ ]:
# orderBy com MÚLTIPLAS colunas: cargo (A-Z) e, dentro de cada cargo, salário decrescente
funcionarios_ordenados = (
    sdf_funcionarios
    .orderBy(col("cargo").asc(), col("salario").desc())
    .select("nome_funcionario", "cargo", "salario")
    .limit(15)
)

# toPandas() converte o resultado já ordenado — pronto para plotar ou exportar
pdf_funcionarios = funcionarios_ordenados.toPandas()
pdf_funcionarios

📌 **Entendendo a saída:**

- `col("valor").desc()` inverteu a ordenação padrão (ascendente) para trazer as vendas de **maior** valor primeiro.
- Com duas colunas em `orderBy`, a primeira (`cargo`) manda na ordenação geral, e a segunda (`salario`) desempata **dentro de cada cargo** — por isso os "Coordenador Comercial" aparecem juntos, já ordenados por salário decrescente.
- `toPandas()` preserva a ordem que veio do Spark — não é preciso reordenar no pandas depois.

🧠 **Regra prática:** sempre `orderBy().limit()` **antes** de `collect()`/`toPandas()`. Ordenar o dataset inteiro sem limitar, e só depois trazer tudo para o Driver, é a receita clássica para um `OutOfMemoryError`.

📎 **`sort()` é um alias de `orderBy()`** — mesma assinatura, mesmo resultado, sem diferença de desempenho. Vamos provar refazendo a ordenação por múltiplas colunas com `sort()` e comparando com o resultado de `orderBy()` de cima.

In [ ]:
# sort() aceita os mesmos argumentos de orderBy() — inclusive múltiplas colunas com asc()/desc()
funcionarios_ordenados_sort = (
    sdf_funcionarios
    .sort(col("cargo").asc(), col("salario").desc())
    .select("nome_funcionario", "cargo", "salario")
    .limit(15)
)

# Comparando com o resultado de orderBy() (pdf_funcionarios, calculado acima) — devem ser idênticos
funcionarios_ordenados_sort.toPandas().equals(pdf_funcionarios)

## Salvando dados: `write`

Até agora só lemos dados prontos da camada Bronze. Mas todo pipeline real também **produz** dados — resultados de transformações que alimentam a próxima camada (Silver/Gold) ou são entregues para outro time/sistema.

O `DataFrameWriter` (acessível via `.write`) é o espelho do `DataFrameReader` (`.read`) que já usamos para ler a Bronze. A diferença fundamental: **`write` é uma ação**, não uma transformação — ela dispara a execução de todo o plano lazy acumulado até ali e materializa o resultado em disco.

🔤 **Save modes** — o que fazer se o destino já existir:

| Modo | Comportamento |
|---|---|
| `overwrite` | Apaga o que existir e escreve do zero |
| `append` | Adiciona aos dados já existentes |
| `error` / `errorifexists` (padrão) | Lança exceção se o destino já existir |
| `ignore` | Não escreve nada se o destino já existir (silenciosamente) |

Vamos gravar dados já calculados anteriormente neste notebook em quatro formatos — `parquet`, `parquet` particionado, `csv` e `json` — para comparar estrutura de arquivos e quando usar cada um.

#### 💡 **Exemplo 7:** Gravando Parquet simples

Vamos persistir `funcionarios_faixa` (calculado no Exemplo 3, já com a coluna `faixa_salarial`) como Parquet — o mesmo formato em que lemos a camada Bronze no início do notebook.

📌 Note o `mode("overwrite")`: sem ele, rodar esta célula uma segunda vez lançaria erro, porque o diretório de destino já existiria.

In [ ]:
import os

caminho_parquet = "../data/output/nb02/funcionarios_faixa"

# write é uma AÇÃO — dispara a execução do pipeline lazy acumulado e grava em disco
(
    funcionarios_faixa
    .write
    .mode("overwrite")          # sobrescreve se já existir
    .parquet(caminho_parquet)
)

# Lendo de volta para provar o round-trip: schema e dados preservados sem esforço extra
funcionarios_faixa_lido = spark.read.parquet(caminho_parquet)
print(f"{funcionarios_faixa_lido.count()} linhas gravadas e recuperadas")
funcionarios_faixa_lido.printSchema()
funcionarios_faixa_lido.show(5)

# O Spark grava um DIRETÓRIO, não um arquivo único — um part-*.parquet por partição
print(os.listdir(caminho_parquet))

📌 **O que aconteceu no disco:**

- O schema (tipos de cada coluna) fica embutido nos arquivos `.parquet` — por isso `read.parquet()` nunca precisa que você declare os tipos manualmente, ao contrário de CSV/JSON.
- Parquet já vem comprimido (Snappy, por padrão) — arquivos bem menores que o CSV/JSON equivalente para o mesmo dado.
- O marcador `_SUCCESS` (arquivo vazio) indica que a escrita terminou **sem falhas** — útil para pipelines automatizados verificarem sucesso antes de prosseguir.

💡 **Dica:** o número de arquivos `part-*.parquet` é igual ao número de partições do DataFrame no momento do `write`. `filter`/`select`/`withColumn` **não** disparam shuffle, então esse número reflete os arquivos lidos originalmente da Bronze, não necessariamente o `spark.sql.shuffle.partitions` (8) configurado na SparkSession.


#### 💡 **Exemplo 8:** Parquet particionado com `partitionBy`

`partitionBy` grava os dados em **subdiretórios por valor** de uma ou mais colunas — o mesmo esquema de particionamento Hive que já existe na própria Bronze: repare que `../data/bronze/vendas` está organizado em pastas `ano=.../mes=.../`.

Vamos particionar `funcionarios_faixa` por `faixa_salarial` — assim, uma consulta futura que filtre por `faixa_salarial` lê **só** a pasta relevante, sem escanear o dataset inteiro (**partition pruning**).

⚠️ **Cuidado:** só particione por colunas de **baixa cardinalidade** (poucos valores distintos, como `faixa_salarial` ou `ano`/`mes`). Particionar por uma coluna de alta cardinalidade (ex.: `id_funcionario`) geraria uma explosão de pastinhas minúsculas — o efeito contrário do que se busca.

In [ ]:
caminho_particionado = "../data/output/nb02/funcionarios_por_faixa"

(
    funcionarios_faixa
    .write
    .mode("overwrite")
    .partitionBy("faixa_salarial")   # uma subpasta por valor distinto de faixa_salarial
    .parquet(caminho_particionado)
)

# Cada valor distinto de faixa_salarial virou uma pasta "faixa_salarial=<valor>"
print(sorted(os.listdir(caminho_particionado)))

📌 **Partition pruning na prática:** ao ler de volta e filtrar pela coluna de partição, o Spark nem chega a abrir os arquivos das outras pastas — ele decide **antes** de ler, só olhando os nomes dos diretórios no plano físico.

In [ ]:
funcionarios_seniors = (
    spark.read.parquet(caminho_particionado)
    .filter(col("faixa_salarial") == "Senior")
)

funcionarios_seniors.show(5)

# explain() revela o plano físico: procure por "PartitionFilters" no scan do Parquet —
# é a prova de que o Spark pulou as pastas Junior/Pleno/Executivo sem sequer lê-las
funcionarios_seniors.explain()

#### 💡 **Exemplo 9:** Exportando CSV para consumo externo

Nem todo consumidor de dados fala Parquet — analistas de negócio costumam pedir CSV para abrir no Excel. Vamos exportar `vendas_grandes` (calculado no bloco de `filter`, vendas > R\$500) como CSV, com cabeçalho.

⚠️ **Atenção:** CSV não guarda schema (tudo vira texto) nem é comprimido por padrão — é o formato mais "universal" dos três, mas também o menos eficiente.

In [ ]:
caminho_csv = "../data/output/nb02/vendas_grandes_csv"

(
    vendas_grandes
    .write
    .mode("overwrite")
    .option("header", True)     # grava o nome das colunas na primeira linha
    .option("sep", ";")         # separador ; — evita conflito com vírgula decimal em pt-BR
    .csv(caminho_csv)
)

arquivos_csv = sorted(f for f in os.listdir(caminho_csv) if f.endswith(".csv"))
print(f"{len(arquivos_csv)} arquivo(s) part-*.csv gerado(s) — um por partição")

📌 Assim como o Parquet, o CSV saiu **um arquivo por partição** do DataFrame no momento do `write` — herdadas da leitura da Bronze, já que `filter()` não faz shuffle. Para um analista que só sabe abrir "um arquivo" no Excel, vários `part-*.csv` são inconvenientes.

💡 **Dica:** `coalesce(1)` força tudo para **uma única partição** antes de escrever — útil para resultados pequenos, destinados a consumo humano. **Nunca** faça isso em datasets grandes: você perde todo o paralelismo da escrita e concentra tudo em um único Executor, criando um gargalo (e um possível `OutOfMemoryError`).

In [ ]:
caminho_csv_unico = "../data/output/nb02/vendas_grandes_arquivo_unico"

(
    vendas_grandes
    .coalesce(1)                # reduz para 1 partição — só compensa para saída pequena!
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(caminho_csv_unico)
)

[f for f in os.listdir(caminho_csv_unico) if f.endswith(".csv")]

#### 💡 **Exemplo 10:** Exportando JSON (JSON Lines)

O último formato: JSON. Vamos gravar `resumo` — os "Vendedor Pleno" que ganham acima da média do próprio cargo (Exemplo 4) — dados que fazem mais sentido como payload de API ou integração entre sistemas do que como planilha.

⚠️ **Atenção:** o Spark grava (e lê) **JSON Lines** — um objeto JSON **por linha**, sem vírgulas nem colchetes envolvendo tudo. Isso permite processar o arquivo linha a linha, em paralelo, sem carregar o arquivo inteiro na memória. Um `.json()` do Spark **não é** um único array JSON gigante.

In [ ]:
caminho_json = "../data/output/nb02/destaques_vendedor_pleno"

(
    resumo
    .write
    .mode("overwrite")
    .json(caminho_json)
)

# Cada linha do arquivo é um objeto JSON independente — dá pra processar com um simples
# "for linha in arquivo", sem precisar carregar/parsear o arquivo inteiro de uma vez
arquivo_json = next(f for f in os.listdir(caminho_json) if f.endswith(".json"))
with open(f"{caminho_json}/{arquivo_json}") as f:
    for linha in f.readlines()[:3]:
        print(linha.strip())

📌 **Parquet vs. CSV vs. JSON — resumo:**

| | Parquet | CSV | JSON (Lines) |
|---|---|---|---|
| Orientação | Colunar | Linha | Linha |
| Schema embutido | ✅ Sim | ❌ Não (tudo texto) | Parcial (tipos primitivos) |
| Compressão padrão | ✅ Snappy | ❌ Nenhuma | ❌ Nenhuma |
| Tamanho em disco | Menor | Maior | Maior |
| Dados aninhados (structs/arrays) | ✅ Sim | ❌ Não | ✅ Sim |
| Legível por humanos | ❌ Não | ✅ Sim | ✅ Sim |
| Quando usar | Entre etapas do pipeline (Bronze → Silver → Gold) | Entrega para Excel/BI | Payload de API, logs, integração entre sistemas |

🧠 **Regra prática:** dentro do seu pipeline de dados, use **Parquet** sempre que possível. Reserve CSV/JSON para as **bordas** do sistema — onde humanos ou sistemas externos (não-Spark) precisam ler o resultado.


In [ ]:
# Encerra a SparkSession e libera recursos (threads, memória)
# Bom prática: sempre parar a sessão ao final do notebook
spark.stop()

---
🎉 **Parabéns!** Você completou o notebook 02 de PySpark.

Você aprendeu:
- Criar uma SparkSession
- Ler arquivos Parquet
- `select`, `withColumn`, `filter`/`where`, `orderBy`/`sort`, `drop`, `distinct`, `describe`
- `udf` e `pandas_udf` — funções definidas pelo usuário, linha a linha e vetorizadas
- A diferença entre `show`, `collect` e `toPandas`
- `write` em Parquet, Parquet particionado (`partitionBy`), CSV e JSON — save modes, `coalesce` e quando usar cada formato
- O conceito de **lazy evaluation** (transformações vs. ações) — e que `write`, assim como `count`/`show`/`collect`, é uma **ação**

▶️ **Próximo:** Notebook 03 — Agregações de Negócio com `groupBy`/`agg`/`orderBy`
